# Activity 8: Ensemble Classifiers on MNIST

**Goal:** Train several classifiers on MNIST (handwritten digits 0–9), combine them into hard and soft voting ensembles, and compare performance.

- **Training set:** 30,000 instances  
- **Test set:** 5,000 instances  
- **Classifiers:** Random Forest, K-Nearest Neighbors, SGD Classifier, Naive Bayes  
- **Ensemble:** Hard voting and soft voting

In [ ]:
# Imports and load MNIST (30k train, 5k test)
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings("ignore")

# Load MNIST (handwritten digits 0-9)
mnist = fetch_openml("mnist_784", as_frame=False, parser="auto")
X, y = mnist.data, mnist.target.astype(int)

# Use 30,000 for training and 5,000 for testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=30_000, test_size=5_000, random_state=42, stratify=y
)
print("Training set:", X_train.shape[0], "| Test set:", X_test.shape[0])

## Part 1 — Implementation: Train individual classifiers

In [ ]:
# Train Random Forest, KNN, SGD, Naive Bayes
rf = RandomForestClassifier(n_estimators=100, random_state=42)
knn = KNeighborsClassifier(n_neighbors=5)
sgd = SGDClassifier(random_state=42, max_iter=1000, loss="log_loss")
nb = GaussianNB()

classifiers = {
    "Random Forest": rf,
    "KNN": knn,
    "SGD Classifier": sgd,
    "Naive Bayes": nb,
}

print("Training individual classifiers...")
scores = {}
for name, clf in classifiers.items():
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)
    acc = accuracy_score(y_test, pred)
    scores[name] = acc
    print(f"  {name}: {acc:.4f}")
print("Done.")

## Ensemble: Hard voting and soft voting

In [ ]:
# Hard voting: majority class wins
voting_hard = VotingClassifier(
    estimators=[("rf", rf), ("knn", knn), ("sgd", sgd), ("nb", nb)],
    voting="hard"
)
voting_hard.fit(X_train, y_train)
pred_hard = voting_hard.predict(X_test)
acc_hard = accuracy_score(y_test, pred_hard)
print("Hard voting accuracy:", f"{acc_hard:.4f}")

# Soft voting: average predicted probabilities (requires probability support)
# Naive Bayes and RF support predict_proba; SGD with loss='log_loss' does; KNN does.
voting_soft = VotingClassifier(
    estimators=[("rf", rf), ("knn", knn), ("sgd", sgd), ("nb", nb)],
    voting="soft"
)
voting_soft.fit(X_train, y_train)
pred_soft = voting_soft.predict(X_test)
acc_soft = accuracy_score(y_test, pred_soft)
print("Soft voting accuracy:", f"{acc_soft:.4f}")

## Compare performance: Ensemble vs individual classifiers

In [ ]:
# Summary table
scores["Hard voting"] = acc_hard
scores["Soft voting"] = acc_soft

print("=" * 50)
print("Test accuracy comparison")
print("=" * 50)
for name, acc in sorted(scores.items(), key=lambda x: -x[1]):
    print(f"  {name:20s} {acc:.4f}")
print("=" * 50)

best_individual = max((scores[n] for n in ["Random Forest", "KNN", "SGD Classifier", "Naive Bayes"]))
best_ensemble = max(acc_hard, acc_soft)
print(f"\nBest individual classifier: {best_individual:.4f}")
print(f"Best ensemble:             {best_ensemble:.4f}")
print(f"\nEnsemble vs best individual: {'better' if best_ensemble >= best_individual else 'same or worse'} (ensemble {best_ensemble:.4f} vs individual {best_individual:.4f})")